## Advanced Rag System

In [3]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 1. Navigate to project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))

load_dotenv()

# 2. Load centralized config & embedding model
from src.services.llm_services import load_config, get_text_embeddings

config = load_config("src/config/config.yaml")
embeddings = get_text_embeddings(config)

print(f"📂 Project Root: {os.getcwd()}")
print(f"✅ Embedding Model: {config['text_emb_provider']} / {config['text_emb_model']}")

c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:133: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📂 Project Root: c:\Users\User\Desktop\Operation Legder-Mind
✅ Embedding Model: sbert / sentence-transformers/all-MiniLM-L6-v2


In [6]:
import weaviate
from langchain_weaviate import WeaviateVectorStore

# 1. Connect to local Weaviate running in Docker
client = weaviate.connect_to_local()
INDEX_NAME = "FinancialReportChunks"

if not client.is_ready():
    raise ConnectionError("❌ Local Weaviate is not running. Start Docker container first!")

# 2. Connect to the existing collection (No re-indexing required!)
vectorstore = WeaviateVectorStore(
    client=client,
    index_name=INDEX_NAME,
    text_key="text",
    embedding=embeddings
)

# 3. Check collection count
collection = client.collections.get(INDEX_NAME)
count = collection.aggregate.over_all(total_count=True).total_count

print(f"✅ Connected to existing Weaviate collection '{INDEX_NAME}'")
print(f"📊 Total indexed chunks available: {count}")

✅ Connected to existing Weaviate collection 'FinancialReportChunks'
📊 Total indexed chunks available: 543


In [7]:
query = "What is Uber's core mission statement as described in the 2024 annual report?"

# Perform similarity search
docs = vectorstore.similarity_search(query, k=3)

print("=" * 60)
print(f"🔍 QUERY: {query}")
print("=" * 60)

for i, doc in enumerate(docs, 1):
    page = doc.metadata.get("page", "N/A")
    splitter = doc.metadata.get("splitter", "N/A")
    
    print(f"\n🎯 [Rank #{i}] (SEC 10-K Page: {page} | Strategy: {splitter})")
    print("-" * 60)
    print(doc.page_content.strip())
    print("-" * 60)

🔍 QUERY: What is Uber's core mission statement as described in the 2024 annual report?

🎯 [Rank #1] (SEC 10-K Page: 1.0 | Strategy: recursive)
------------------------------------------------------------
Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who are relentless about our  
mission to help people go anywhere and get anything and earn their way. 
Movement is what we power. It’s our lifeblood. It runs through our veins. It’s 
what gets us out of bed each morning. It pushes us to constantly reimagine 
how we can move better. For you. For all the places you want to go. For all the 
things you want to get. For all the ways you want to earn. Across the entire 
world. In real time. At the incredible speed of now.
------------------------------------------------------------

🎯 [Rank #2] (SEC 10-K Page: 11.0 | Strategy: recursive)
------------------------------------------------------------
Employees have access to an in

## Hybrid Search(Dense(vector search) +Sparce(BM25))

In [8]:
from collections import defaultdict
from langchain_core.documents import Document


# 1. Dense Search using Weaviate
def dense_search(query: str, top_k: int = 8):
    """Search using Dense Vectors (SBERT)."""
    query_vector = embeddings.embed_query(query)
    response = collection.query.near_vector(
        near_vector=query_vector,
        limit=top_k,
        return_metadata=["distance"]
    )
    
    results = []
    for obj in response.objects:
        doc = Document(
            page_content=obj.properties.get("text", ""),
            metadata={
                "page": obj.properties.get("page"),
                "source": obj.properties.get("source"),
                "splitter": obj.properties.get("splitter"),
            }
        )
        results.append({
            "doc": doc,
            "score": float(obj.metadata.distance) if obj.metadata else 0.0,
            "doc_id": str(obj.uuid)
        })
    return results


# 2. Sparse Search using Weaviate's Built-in BM25
def bm25_search(query: str, top_k: int = 8):
    """Search using Weaviate's native BM25 sparse keyword engine."""
    response = collection.query.bm25(
        query=query,
        limit=top_k,
        return_metadata=["score"]
    )
    
    results = []
    for obj in response.objects:
        doc = Document(
            page_content=obj.properties.get("text", ""),
            metadata={
                "page": obj.properties.get("page"),
                "source": obj.properties.get("source"),
                "splitter": obj.properties.get("splitter"),
            }
        )
        results.append({
            "doc": doc,
            "score": float(obj.metadata.score) if obj.metadata else 0.0,
            "doc_id": str(obj.uuid)
        })
    return results

# 3. Reciprocal Rank Fusion (RRF) matching your dictionary format
def rrf_fusion(dense_results, bm25_results, k: int = 60, top_k: int = 3):
    """
    Fuse dense and sparse results using RRF.
    Returns: List of dicts with 'doc', 'score', and 'doc_id'
    """
    rrf_scores = defaultdict(float)
    doc_lookup = {}
    # Rank Dense
    for rank, item in enumerate(dense_results, 1):
        doc_id = item["doc_id"]
        doc_lookup[doc_id] = item["doc"]
        rrf_scores[doc_id] += 1.0 / (k + rank)
    # Rank BM25
    for rank, item in enumerate(bm25_results, 1):
        doc_id = item["doc_id"]
        doc_lookup[doc_id] = item["doc"]
        rrf_scores[doc_id] += 1.0 / (k + rank)
    # Sort descending by RRF score
    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    fused = []
    for doc_id, score in sorted_items[:top_k]:
        fused.append({
            "doc": doc_lookup[doc_id],
            "score": score,
            "doc_id": doc_id
        })
    return fused

In [9]:
query = "What is Uber's core mission statement as described in the 2024 annual report?"
# 1. Run Dense and BM25 searches
dense_results = dense_search(query, top_k=8)
bm25_results = bm25_search(query, top_k=8)
# 2. Test hybrid fusion
fused_results = rrf_fusion(dense_results, bm25_results, k=60, top_k=3)
# 3. Print results (your exact format)
print(f"🔀 Hybrid (RRF) search: '{query}'\n")
for i, res in enumerate(fused_results, 1):
    page = res['doc'].metadata.get('page', 'N/A')
    print(f"  [{i}] (RRF: {res['score']:.4f} | Page: {page}) {res['doc'].page_content[:120]}...")

🔀 Hybrid (RRF) search: 'What is Uber's core mission statement as described in the 2024 annual report?'

  [1] (RRF: 0.0328 | Page: 1.0) Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who a...
  [2] (RRF: 0.0161 | Page: 11.0) Employees have access to an internal jobs marketplace for full-time jobs as well as short-term stretch assignments that ...
  [3] (RRF: 0.0161 | Page: 141.0) f 2021 Proxy statement
1725 3rd Street
San Francisco, California 94158
uber.com...


## Reranking

In [12]:
from sentence_transformers import CrossEncoder

# 1. Load the Cross-Encoder model
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. Extract candidate documents using ["doc"] (Fixes KeyError: 0)
candidate_docs = [res["doc"] if isinstance(res, dict) else res[0] for res in fused_results]

# 3. Create (query, passage) pairs for the cross-encoder
pairs = [[query, doc.page_content] for doc in candidate_docs]

# 4. Compute cross-encoder relevance scores
cross_scores = reranker.predict(pairs)

# 5. Sort candidates by score descending
reranked_results = sorted(
    zip(candidate_docs, cross_scores),
    key=lambda x: x[1],
    reverse=True
)

# 6. Display final top 3 reranked documents
top_k = config.get("rerank_top_n", 3)
print("=" * 65)
print(f"🎯 CROSS-ENCODER RE-RANKED RESULTS (TOP {top_k}):")
print("=" * 65)

for rank, (doc, ce_score) in enumerate(reranked_results[:top_k], 1):
    page = doc.metadata.get("page", "N/A")
    print(f"\n🥇 [Final Rank #{rank}] | Cross-Encoder Score: {ce_score:.4f} | Page: {page}")
    print("-" * 65)
    print(doc.page_content.strip()[:250] + "...")
    print("-" * 65)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

🎯 CROSS-ENCODER RE-RANKED RESULTS (TOP 3):

🥇 [Final Rank #1] | Cross-Encoder Score: 0.4622 | Page: 1.0
-----------------------------------------------------------------
Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who are relentless about our  
mission to help people go anywhere and get anything and earn their way. 
Movement is what we power. It’s...
-----------------------------------------------------------------

🥇 [Final Rank #2] | Cross-Encoder Score: -2.8720 | Page: 11.0
-----------------------------------------------------------------
Employees have access to an internal jobs marketplace for full-time jobs as well as short-term stretch assignments that enable them to 
have an impact on other areas of the business. Our goal is to he lp our employees be their best selves by providin...
-----------------------------------------------------------------

🥇 [Final Rank #3] | Cross-Encoder Score: -5.8232 | Page: 141

In [13]:
from collections import defaultdict
from typing import Dict, Any, List
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from src.services.llm_services import get_llm

# 1. Initialize Cross-Encoder Reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. Initialize LLM
try:
    llm = get_llm(config)
except Exception:
    llm = None


def rrf_fusion_scores(dense_results: List[Document], bm25_results: List[Document], k: int = 60) -> List[Dict[str, Any]]:
    """Reciprocal Rank Fusion (RRF): score(d) = sum(1 / (k + rank(d)))"""
    rrf_scores = defaultdict(float)
    doc_map = {}

    for rank, doc in enumerate(dense_results, 1):
        doc_id = doc.metadata.get("page_content_hash") or hash(doc.page_content)
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)

    for rank, doc in enumerate(bm25_results, 1):
        doc_id = doc.metadata.get("page_content_hash") or hash(doc.page_content)
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)

    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [{"doc": doc_map[doc_id], "rrf_score": score, "doc_id": doc_id} for doc_id, score in sorted_items]


def query_librarian(
    question: str,
    top_k_retrieval: int = 10,
    top_k_rerank: int = 3,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    'The Librarian' - Advanced Hybrid RAG Pipeline
    Constraint: query_librarian(question)
    """
    if not client.is_connected():
        client.connect()
    collection = client.collections.get("FinancialReportChunks")

    # 1. HYBRID RETRIEVAL (Dense + BM25)
    # A. Dense SBERT Search
    query_vector = embeddings.embed_query(question)
    dense_res = collection.query.near_vector(near_vector=query_vector, limit=top_k_retrieval, return_metadata=["distance"])
    dense_docs = [
        Document(
            page_content=obj.properties.get("text", ""),
            metadata={"page": obj.properties.get("page"), "source": obj.properties.get("source"), "splitter": obj.properties.get("splitter")}
        )
        for obj in dense_res.objects
    ]

    # B. Sparse BM25 Keyword Search
    bm25_res = collection.query.bm25(query=question, limit=top_k_retrieval, return_metadata=["score"])
    bm25_docs = [
        Document(
            page_content=obj.properties.get("text", ""),
            metadata={"page": obj.properties.get("page"), "source": obj.properties.get("source"), "splitter": obj.properties.get("splitter")}
        )
        for obj in bm25_res.objects
    ]

    # 2. REFINEMENT 1: Reciprocal Rank Fusion (RRF)
    rrf_candidates = rrf_fusion_scores(dense_docs, bm25_docs, k=60)[:top_k_retrieval]

    # 3. REFINEMENT 2: Cross-Encoder Neural Re-ranking
    candidate_docs = [item["doc"] for item in rrf_candidates]
    ce_pairs = [[question, doc.page_content] for doc in candidate_docs]
    ce_scores = reranker.predict(ce_pairs)

    reranked_pool = []
    for item, ce_score in zip(rrf_candidates, ce_scores):
        reranked_pool.append({
            "doc": item["doc"],
            "rrf_score": item["rrf_score"],
            "cross_encoder_score": float(ce_score),
            "page": item["doc"].metadata.get("page"),
        })

    reranked_pool.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    final_passages = reranked_pool[:top_k_rerank]

    # 4. GROUNDED FINANCIAL GENERATION
    context_blocks = [f"[SEC Form 10-K Page {p['page']}]:\n{p['doc'].page_content.strip()}" for p in final_passages]
    formatted_context = "\n\n".join(context_blocks)

    system_instruction = (
        "You are 'The Librarian', an authoritative financial analyst AI specializing in SEC Form 10-K filings. "
        "Answer the question accurately, concisely, and factually based strictly on the provided context. "
        "Always cite the relevant SEC 10-K page numbers for financial metrics and claims. "
        "If the answer cannot be determined directly from the context, state: 'Based on the provided document, this information is not available.'"
    )

    prompt = (
        f"{system_instruction}\n\n"
        f"--- FINANCIAL DOCUMENT EVIDENCE ---\n{formatted_context}\n\n"
        f"--- INQUIRY ---\n{question}\n\n"
        f"--- AUTHORITATIVE ANSWER ---"
    )

    if llm:
        try:
            response = llm.invoke(prompt)
            answer = response.content if hasattr(response, "content") else str(response)
        except Exception as e:
            answer = f"⚠️ LLM generation failed: {e}"
    else:
        answer = f"Evidence from SEC 10-K Page {final_passages[0]['page']}:\n{final_passages[0]['doc'].page_content.strip()[:300]}..."

    if verbose:
        print("=" * 70)
        print("📚 'THE LIBRARIAN' ADVANCED RAG PIPELINE")
        print("=" * 70)
        print(f"❓ Question: {question}")
        print(f"🔍 Hybrid Search:  {len(dense_docs)} Dense + {len(bm25_docs)} BM25 results")
        print(f"🔀 RRF Refinement: Merged into {len(rrf_candidates)} candidates")
        print(f"🎯 Cross-Encoder:  Filtered to Top {len(final_passages)} passages")
        print("-" * 70)
        print(f"💡 ANSWER:\n{answer.strip()}\n")
        print("-" * 70)
        print("📑 RE-RANKED CITATIONS:")
        for rank, p in enumerate(final_passages, 1):
            print(f"  [{rank}] Page {p['page']} | CE Score: {p['cross_encoder_score']:+.4f} | RRF Score: {p['rrf_score']:.5f}")
        print("=" * 70)

    return {
        "question": question,
        "answer": answer.strip(),
        "sources": [p["page"] for p in final_passages],
        "reranked_chunks": final_passages,
    }

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [14]:
# Test 1: Numerical entity & Gross Bookings
result_1 = query_librarian("What was Uber's total Gross Bookings reported for the fiscal year 2024?")
# Test 2: Specific corporate program entity
result_2 = query_librarian("What benefits do Uber One members receive according to the report?")

📚 'THE LIBRARIAN' ADVANCED RAG PIPELINE
❓ Question: What was Uber's total Gross Bookings reported for the fiscal year 2024?
🔍 Hybrid Search:  10 Dense + 10 BM25 results
🔀 RRF Refinement: Merged into 10 candidates
🎯 Cross-Encoder:  Filtered to Top 3 passages
----------------------------------------------------------------------
💡 ANSWER:
⚠️ LLM generation failed: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402, 'metadata': {'limit_source': 'openrouter_credits', 'remedy_hint': 'Add credits at https://openrouter.ai/settings/credits, or lower max_tokens / prompt size to fit your remaining balance.'}}}

----------------------------------------------------------------------
📑 RE-RANKED CITATIONS:
  [1] Page 63.0 | CE Score: +4.5517 | RRF Score: 0.03101
  [2] Page 57.0 | CE Score: +1.4412 | RRF Score: 0.01587